# Quantitative Finance Project: Updated S&P 500 Price Data

## Purpose

Download the latest version of Kaggle's [S&P 500 Stocks: 25 Years of Data (Updated Daily)](https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily), clean its daily price and company files, and save an analysis-ready dataset for the EDA and machine-learning notebooks.

The Kaggle dataset is maintained by its contributor and is described as updated daily. This notebook forces a fresh download when run. Kaggle account authentication is required in Colab. Price history covers the current S&P 500 sample; using current constituents in earlier years creates survivorship bias.

## 1. Connect Colab and download the latest dataset version

Set up Kaggle authentication in Colab (KaggleHub may prompt for your Kaggle credentials or use a configured Kaggle API token). Do not put a private token directly into a notebook you plan to share.

In [ ]:
%pip -q install kagglehub pyarrow
from pathlib import Path
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DATASET_HANDLE = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
# force_download requests the current Kaggle version rather than relying on a local cache.
DATASET_DIR = Path(kagglehub.dataset_download(DATASET_HANDLE, force_download=True))
OUTPUT_DIR = Path('/content/drive/MyDrive/sp500_kaggle_processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Downloaded dataset directory:',DATASET_DIR)
print('Output directory:',OUTPUT_DIR)

### Why this download step matters

The KaggleHub call requests the current dataset version instead of using a cached copy. The folder path confirms where KaggleHub placed the source files. If authentication fails, configure your Kaggle credentials in Colab and rerun this cell.

## 2. Locate source files and inspect their schemas

In [ ]:
def find_csv(root, expected_name):
    matches=list(root.rglob(expected_name))
    if not matches: raise FileNotFoundError(f'{expected_name} not found under {root}')
    return matches[0]

STOCKS_CSV=find_csv(DATASET_DIR,'sp500_stocks.csv')
COMPANIES_CSV=find_csv(DATASET_DIR,'sp500_companies.csv')
print('Prices:',STOCKS_CSV)
print('Companies:',COMPANIES_CSV)
print('Price columns:',pd.read_csv(STOCKS_CSV,nrows=0).columns.tolist())
print('Company columns:',pd.read_csv(COMPANIES_CSV,nrows=0).columns.tolist())
prices_raw=pd.read_csv(STOCKS_CSV,low_memory=False)
companies_raw=pd.read_csv(COMPANIES_CSV,low_memory=False)
print('Raw price rows:',len(prices_raw),'| company rows:',len(companies_raw))

### What to check in the source schema

The price table should provide a date, symbol, OHLC prices, and volume; adjusted close may be present or absent. The company file should provide symbol metadata such as sector. These checks catch a renamed or changed Kaggle file before cleaning begins.

## 3. Standardize and clean daily prices

Keep a stable set of fields, parse dates and numbers, remove rows without a usable symbol/date/close, and retain one row per symbol and trading date. Use adjusted close for returns when the downloaded version supplies it; otherwise the later notebooks fall back to close and label that limitation.

In [ ]:
def normalize_columns(frame):
    frame=frame.copy()
    frame.columns=[str(c).strip().lower().replace(' ','_').replace('-','_') for c in frame.columns]
    aliases={'ticker':'symbol','ticker_symbol':'symbol','adjusted_close':'adj_close','adjclose':'adj_close',
             'adj._close':'adj_close','date_time':'date'}
    return frame.rename(columns={c:aliases.get(c,c) for c in frame.columns})

p=normalize_columns(prices_raw)
required={'date','symbol','open','high','low','close','volume'}
missing=required-set(p.columns)
if missing: raise ValueError(f'Kaggle price schema is missing {sorted(missing)}; columns={p.columns.tolist()}')
if 'adj_close' not in p.columns: p['adj_close']=np.nan
rows_in=len(p)
p['date']=pd.to_datetime(p['date'],errors='coerce')
for col in ['open','high','low','close','adj_close','volume']:
    p[col]=pd.to_numeric(p[col],errors='coerce')
p['symbol']=p['symbol'].astype('string').str.strip().str.upper()
invalid_dates=int(p['date'].isna().sum())
invalid_close=int(p['close'].isna().sum())
p=p.dropna(subset=['date','symbol','close'])
p=p[p['close']>0]
duplicate_rows=int(p.duplicated(['symbol','date'],keep='last').sum())
p=p.drop_duplicates(['symbol','date'],keep='last').sort_values(['symbol','date']).reset_index(drop=True)
if not p.empty and p['adj_close'].notna().all():
    p['return_price']=p['adj_close']
    price_basis='adj_close'
else:
    p['return_price']=p['close']
    price_basis='close (adjusted close unavailable or incomplete)'
p['daily_return']=p.groupby('symbol')['return_price'].pct_change()
print(f'Rows in: {rows_in:,} | cleaned: {len(p):,} | invalid dates: {invalid_dates:,} | invalid close: {invalid_close:,} | duplicate keys removed: {duplicate_rows:,}')
print('Price field used consistently for returns:',price_basis)
display(p.head())

### Why these cleaning rules are used

Each retained row represents one symbol on one trading date. Rows with unusable dates, symbols, or closing prices cannot support return calculations, so they are removed. Duplicate symbol-date rows are reduced to one record. Returns use adjusted close only when it is complete; otherwise they use close consistently across the sample.

## 4. Clean company metadata and attach sector labels

In [ ]:
c=normalize_columns(companies_raw)
if 'symbol' not in c.columns:
    raise ValueError(f'No symbol column in company metadata: {c.columns.tolist()}')
c['symbol']=c['symbol'].astype('string').str.strip().str.upper()
for col in c.select_dtypes(include=['object','string']).columns:
    c[col]=c[col].astype('string').str.strip()
c=c.dropna(subset=['symbol']).drop_duplicates('symbol',keep='last')
# Normalize common names to convenient labels while retaining original columns.
rename={}
for col,target in [('company','company'),('security','company'),('sector','sector'),('gics_sector','sector'),
                   ('sub_industry','sub_industry'),('gics_sub_industry','sub_industry')]:
    if col in c.columns and target not in c.columns: rename[col]=target
c=c.rename(columns=rename)
if 'sector' not in c.columns: c['sector']='Unknown'
p=p.merge(c[['symbol','sector']].drop_duplicates('symbol'),on='symbol',how='left')
p['sector']=p['sector'].fillna('Unknown')
print('Unique price symbols:',p['symbol'].nunique(),'| metadata symbols:',c['symbol'].nunique())
display(c.head())

### How to read the metadata merge

The left join keeps the price history even when a symbol has no matching company metadata. Such rows receive `Unknown` sector. Check the displayed symbol counts and sample rows to confirm that sector labels were attached as expected.

## 5. Save the clean analysis dataset and preprocessing report

In [ ]:
PRICE_PATH=OUTPUT_DIR/'sp500_prices.parquet'
COMPANY_PATH=OUTPUT_DIR/'sp500_companies.parquet'
REPORT_PATH=OUTPUT_DIR/'preprocessing_report.csv'
p.to_parquet(PRICE_PATH,index=False)
c.to_parquet(COMPANY_PATH,index=False)
report=pd.DataFrame([
 {'metric':'source_rows','value':rows_in},
 {'metric':'clean_rows','value':len(p)},
 {'metric':'symbols','value':p['symbol'].nunique()},
 {'metric':'first_date','value':p['date'].min()},
 {'metric':'last_date','value':p['date'].max()},
 {'metric':'invalid_dates_removed','value':invalid_dates},
 {'metric':'invalid_close_rows_before_filter','value':invalid_close},
 {'metric':'duplicate_symbol_date_rows_removed','value':duplicate_rows},
 {'metric':'missing_adjusted_close_rows','value':int(p['adj_close'].isna().sum())},
])
report.to_csv(REPORT_PATH,index=False)
display(report)
print('Saved:',PRICE_PATH,COMPANY_PATH,REPORT_PATH)

### What the saved files contain

The Parquet price file contains cleaned daily rows plus the return-price field and sector. The company file holds cleaned metadata, while the CSV report records row counts, date coverage, missing adjusted prices, and duplicate removals. The reported last date tells you how current this download is.

## 6. Quick coverage checks

In [ ]:
print('Daily observations per symbol (summary):')
display(p.groupby('symbol').agg(first_date=('date','min'),last_date=('date','max'),rows=('date','size')).describe(include='all'))
print('Rows by sector:')
display(p.groupby('sector')['symbol'].nunique().sort_values(ascending=False).rename('symbols').to_frame())
fig,axes=plt.subplots(1,2,figsize=(14,5))
p.groupby('date').size().plot(ax=axes[0],color='#4472C4')
axes[0].set(title='Number of symbols with prices each day',xlabel='Date',ylabel='Symbols');axes[0].grid(alpha=.25)
p.groupby('sector')['symbol'].nunique().sort_values().plot(kind='barh',ax=axes[1],color='#70AD47')
axes[1].set(title='Symbols per sector',xlabel='Symbols',ylabel='');axes[1].grid(axis='x',alpha=.25)
fig.tight_layout();plt.show()

### Reading the coverage charts

The first chart shows how many symbols have observations on each date; uneven counts can reflect listing history or incomplete source coverage. The second chart shows the number of symbols represented by sector. These charts describe the downloaded sample, not official index weights or historical S&P 500 membership.

## Output and caveats

The EDA and modeling notebooks read `sp500_prices.parquet` from the Drive folder above. The Kaggle dataset is community maintained; verify the last date and column names after each refresh. Adjusted prices are preferred for returns because they account for corporate actions when available. Historical coverage for today's S&P 500 symbols does not recreate past index membership.

# Detailed field guide and preprocessing rationale

## What one price row represents

A price row represents one listed symbol on one trading date. The same symbol has a sequence of rows over time; the same date has many symbols. Keeping both `symbol` and `date` is essential: the pair identifies the observation. Sorting by this pair before any return or rolling-window calculation prevents a value from one company being used in another company's series.

| Field | Meaning | How this notebook uses it |
|---|---|---|
| `date` | Trading session date | Parses to a date type and orders each time series. |
| `symbol` | Exchange ticker | Normalized to uppercase and used as the group key. |
| `open` | Opening price | Preserved for later market studies; not used in the current target. |
| `high` | Highest price during the session | Preserved for intraday-range features. |
| `low` | Lowest price during the session | Preserved for intraday-range features. |
| `close` | Closing price in the source file | Fallback return price if adjusted close is absent or incomplete. |
| `adj_close` | Close adjusted for corporate actions when provided | Preferred return price when complete across the downloaded file. |
| `volume` | Shares traded during the session | Preserved for trading-activity features. |
| `sector` | Company sector from the companion file | Joined by symbol and used as metadata and an ML feature. |
| `return_price` | Consistent price series selected for this run | Keeps return calculations on one price basis. |
| `daily_return` | Percentage change in `return_price` | Derived with `pct_change()` within symbol. |

Column names in Kaggle versions can change capitalization or spacing. The normalization step standardizes these names before checking required columns. The schema check is deliberate: a missing required field should stop processing with a useful message instead of silently producing invalid features.

## Why each cleaning decision is made

1. **Trim and normalize ticker strings.** A trailing space or lower-case symbol can prevent a valid join. Normalization makes joins reliable while retaining the symbol identity.
2. **Parse dates with errors coerced to missing.** Bad date strings become visible as missing values and are counted before removal.
3. **Convert prices and volume to numeric types.** Text-encoded numbers can otherwise break return calculations or sort lexicographically.
4. **Remove rows without date, symbol, or close.** Such rows cannot be placed in a time series or used to calculate returns.
5. **Require positive closing prices.** Zero and negative prices are invalid for ordinary listed-equity return calculations.
6. **Resolve duplicate symbol-date observations.** The notebook retains the last source row for a duplicate key and reports the number removed. If duplicate rows disagree, inspect the source rather than assuming the last value is correct.
7. **Choose one price basis.** Adjusted close accounts for corporate actions when fully present. If it is missing or incomplete, close is used for all observations so a single company's return series does not mix adjusted and unadjusted prices.
8. **Keep the company metadata join left-sided.** Price history remains available when a company label is missing. Unmatched rows are identified as `Unknown` instead of disappearing from the market dataset.

## Quality indicators in the report

- **Source rows and clean rows:** quantify how many observations remain after date, symbol, close, and duplicate rules.
- **Unique symbols:** show how broad the data file is, but do not prove that all historical index members are present.
- **First and last date:** establish the effective historical window for this exact Kaggle version.
- **Missing adjusted close rows:** indicate whether the adjustment field is complete enough to use consistently.
- **Duplicate symbol-date rows removed:** reveal repeated observations that could otherwise overweight a date.
- **Rows by sector:** describe the composition of the current metadata join; these are company counts, not market-cap weights.

A higher clean-row count is not automatically better. A dataset can have many rows and still contain stale symbols, bad corporate-action adjustments, or incomplete historical constituents. Review both the counts and representative rows.

## Repeatable refresh procedure

1. Run the KaggleHub download cell to retrieve the newest available version.
2. Record the printed first and last dates and the Kaggle dataset URL in your lab notes.
3. Run schema checks before cleaning. If the contributor changes a file or column name, update the aliases only after examining the new header.
4. Run cleaning and save the Parquet outputs to Drive.
5. Review the report and coverage charts before running the EDA notebook.
6. When refreshing data, regenerate downstream features and model splits. Do not combine an old feature panel with a new price file.

The downloaded data is a snapshot of the Kaggle contributor's current data. The dataset page describes it as updated daily, but the notebook still checks its actual last date on each run. Do not assume that the update completed for a market holiday or that all recent observations are final.

## Troubleshooting notes

- **KaggleHub authentication error:** configure Kaggle credentials in the Colab environment; never paste a private token into a public notebook.
- **Expected CSV not found:** inspect `DATASET_DIR.rglob('*.csv')` and update the expected file names if Kaggle changes its folder structure.
- **Required price field missing:** inspect the printed schema and confirm the source file still represents daily OHLCV data.
- **No sector column:** the price file can still be analyzed, but sector comparisons and sector features need a metadata source.
- **No adjusted close:** the notebook falls back to close and reports that choice. This may leave dividend effects out of returns.
- **Unexpected duplicate count:** inspect duplicate keys and compare source rows before changing the keep-last rule.
- **Storage or memory pressure:** use Parquet for later steps and avoid keeping multiple full CSV copies in memory. The Kaggle files may be substantially larger than the compressed download.

## Research boundary

This preparation notebook creates daily market observations. It does not contain SEC filing facts, EBITDA, point-in-time market capitalization, or historical S&P 500 membership. The downstream project predicts next-session return direction from lagged price and volume features. A model trained on a currently represented constituent universe can overstate historical performance because firms that left the index may be absent.

## Reading the coverage charts carefully

The symbol-count time series is a coverage diagnostic. If the count changes abruptly, possible causes include initial listing dates, missing source rows, mergers, ticker changes, or contributor collection issues. A decline near the most recent date can also indicate that the dataset update has not finished. It is not evidence that the S&P 500 itself contained fewer constituents that day.

The sector chart counts symbols once per sector using the joined company table. It treats a small firm and a large firm equally. Do not interpret it as sector allocation, market capitalization, index weight, or return contribution. For a weighted sector portfolio, market capitalization or official index weights would be required.

Before relying on either chart, compare the displayed dates to the project question and save the report. Data freshness is part of the method: the Kaggle file can change between runs even when the code does not.

# Preprocessing walkthrough and submission notes

## Data lineage: source to downstream model

| Stage | Input | Operation | Output | Reason for the stage |
|---|---|---|---|---|
| Refresh | Kaggle dataset handle | Download latest version | Kaggle cache folder | Avoid relying on an old local snapshot. |
| Discovery | Dataset folder | Locate the two expected CSV files | Stable file paths | Fail early if the source layout changes. |
| Price parse | Daily price CSV | Normalize column names and types | Tidy price table | Make calculations consistent and machine readable. |
| Key cleanup | Date and symbol | Remove invalid keys and duplicate pairs | One row per symbol-date | Prevent malformed observations from corrupting time order. |
| Price basis | Close and adjusted close | Select one complete basis | `return_price` | Keep returns comparable over corporate actions when possible. |
| Metadata join | Company table | Join by normalized symbol | Sector label on each price row | Enable sector summaries without dropping price history. |
| Save | Clean tables | Write Parquet and CSV report | Files in Drive | Reuse data without repeating CSV parsing. |
| Audit | Clean outputs | Plot symbol coverage and sector counts | Coverage charts | Reveal omissions and changing representation. |

The separation between these stages helps isolate problems. If the last date is unexpectedly old, inspect the downloaded snapshot before debugging feature code. If the sector chart has many `Unknown` rows, investigate symbol normalization or metadata coverage before model fitting. If the daily row count changes after a refresh, compare the report before concluding that the market itself changed.

## How a duplicate key can affect finance calculations

A repeated `(symbol, date)` row can create two returns for one session. If both rows survive, the next `pct_change()` can treat them as consecutive observations even though their dates are equal. That creates a zero-length interval and may alter the return after it. The notebook reports and removes duplicate keys before calculating returns. For a research dataset, it is still good practice to inspect a few duplicate source rows when the duplicate count is nonzero: a duplicate may be a true repeated import, a correction, or a record with a different adjustment basis.

The current keep-last rule is a pragmatic deduplication policy, not a claim that the last row is always economically correct. If the contributor later documents an official correction field or data revision timestamp, use that source rule instead. Preserve an untouched copy of the original CSV if you need to audit a cleaning decision later.

## Why a left join is used for sectors

A sector label is useful metadata, but it is not a prerequisite for computing price returns. An inner join would silently discard a stock whenever the metadata file did not include its ticker. A left join keeps that stock's market history and makes its classification gap visible. The `Unknown` category can be reviewed or imputed downstream. Never silently infer a sector from a ticker string.

The company file may be a current snapshot. If a company changed sector or ticker over time, applying today's label to every historic price row can leak future classification information into a historical study. For this project's baseline, sector is a coarse explanatory feature; for a point-in-time backtest, historical sector classifications would be needed.

## Reproducibility record

For each saved run, record the following in your lab notebook or report:

- Kaggle dataset URL and dataset owner.
- Download date and latest trading date printed by this notebook.
- Names and columns of the source files.
- Number of source rows, cleaned rows, symbols, and duplicate keys removed.
- Whether adjusted close was complete and selected for returns.
- Output folder and filenames.
- Any manual changes to aliases, filters, or the duplicate rule.

The dataset updates, so running the same notebook at a later date can produce a different row count and last date. A code version alone is not enough to reproduce a result; pair it with a dataset version or a saved snapshot.

## Pre-submission review checklist

- [ ] Kaggle authentication is configured without exposing a token in the notebook.
- [ ] The downloaded dataset handle is the intended S&P 500 daily dataset.
- [ ] The printed schema includes the fields the project actually uses.
- [ ] The cleaning report's date range makes sense for the download date.
- [ ] The number of duplicate keys removed is reviewed.
- [ ] The adjusted-close availability and return-price basis are written down.
- [ ] Unknown sector values are reviewed rather than silently dropped.
- [ ] The two Parquet files and report are visible in the intended Drive folder.
- [ ] The EDA and ML notebooks read those exact prepared files.

## Sample narrative format for the report

After running the notebook, write a short factual paragraph using the actual printed figures: “The refreshed Kaggle snapshot contained ___ price rows across ___ symbols from ___ through ___. Cleaning removed ___ rows with invalid date/symbol/close information and ___ duplicate symbol-date keys. The project used ___ for return calculations because adjusted-close coverage was ___. Sector metadata matched ___ symbols, with ___ rows classified as Unknown. The output is saved as Parquet for the EDA and model notebooks.” Fill the blanks from the generated report; do not estimate them from a prior version.

This paragraph documents preparation, not a financial finding. It should remain separate from interpretation of stock returns or model performance.

## Practical run-through: from Kaggle refresh to a checked output

When you run the notebook, treat each output as a checkpoint. First confirm that the KaggleHub cell returns a dataset directory. Next compare the printed CSV headers to the expected daily-price and company fields. After cleaning, check that the first/last dates and number of symbols are plausible. Finally confirm the Parquet files and report appear in Drive before opening the EDA notebook.

If one checkpoint fails, fix that stage and rerun downstream cells. For example, a changed column label should be handled in `normalize_columns`; it should not be “fixed” by removing the required-column check. A missing sector should be diagnosed in the metadata file; it should not be fixed by dropping otherwise valid price observations.

### Why the chart output should be saved

The two coverage plots summarize the refreshed data version. Once you run the notebook, save it so the figures and printed report remain with the code. This makes the submission easier to review: a reader sees the method, the data coverage, and the resulting diagnostic chart in one place. Remember that a rerun after the dataset refresh can change the last date and counts.

## Final preparation hand-off to the next notebook

The output contract for downstream notebooks is `sp500_prices.parquet` with `date`, `symbol`, OHLC, volume, `return_price`, `daily_return`, and sector. Keep these field names stable when editing this notebook. If you change a name, update both the EDA and model notebooks and rerun feature generation from the refreshed file. A saved Parquet table represents a specific download snapshot; its filename alone does not identify which Kaggle version produced it. Record the run's date coverage and preprocessing report together with submitted results.